# 01 · AiiDA smoke test + query cookbook

End-to-end Si-bulk SCF on PD-A demonstrating the full Phase A schema, plus the reusable AiiDA recipes: process-state formatting, output parsing, the 5 canonical QueryBuilder patterns, selective HPC cleanup, the Parquet-row preview, and node teardown. This is the canonical 'how the AiiDA layer works end to end' reference.

*Consolidated aiida-scf.ipynb (2026-06-16 notebook cleanup).*

In [12]:
# Smoke test: Si bulk SCF on PD-A, demonstrating the full Phase A schema.
# - StructureData carries provenance extras (source_db / source_id)
# - WorkChainNode carries sweep state + decision cache + lifecycle
# - Cutoff sweep schedule with concrete Ry values
# - Query examples (QueryBuilder by extras / Group)
# - HPC cleanup after parsing

from aiida import load_profile, orm
from aiida.engine import submit
from aiida.orm import (
    Code, Group, InstalledCode, QueryBuilder, StructureData,
    WorkChainNode, load_code, load_group, load_node,
)
from aiida_quantumespresso.common.types import ElectronicType, SpinType
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain

load_profile();


In [13]:
# Si bulk diamond, lattice constant 5.43 A.
# Provenance extras live on StructureData so any downstream WorkChain can find
# (source_db, source_id) via inputs.structure.base.extras (per aiida-analyser pattern).

from ase.build import bulk

ase_si = bulk('Si', 'diamond', a=5.43)
si = StructureData(ase=ase_si).store()

si.base.extras.set_many({
    'source_db': 'smoke-test',
    'source_id': 'si-bulk-diamond-a543',
    # MC3D-specific audit extras would be set by mc3d.py at ingest, e.g.:
    #   'mc3d_total_magnetization': 0.0,
    #   'mc3d_absolute_magnetization': 0.0,
    #   'mc3d_afm_likely': False,
})

print(f'StructureData PK={si.pk}  formula={si.get_formula()}')
print(f'  extras: {dict(si.base.extras.all)}')


StructureData PK=34011  formula=Si2
  extras: {'source_db': 'smoke-test', 'source_id': 'si-bulk-diamond-a543', '_aiida_hash': 'c88e1f47269e96b7949b2f106ed69511bba1f83580e3d92df5c2f3b0b44ec60d'}


In [14]:
# Cutoff schedule: start from family_recommended ecutwfc, step +5 Ry until
# 3 consecutive ΔE/atom < 1 meV. For Si on PD-A, the recommended cutoff is ~30 Ry.

CUTOFF_STEP_RY = 5.0
ECUTRHO_RATIO  = 4.0  # NC pseudo

code = load_code('qe-7.2-pw@scarf')
pseudo_group = load_group('PseudoDojo/0.4/PBEsol/SR/standard/upf')

# Pull each element's recommended cutoff from the pseudo (PseudoDojo ships these
# in pseudo.cutoffs_recommended; aiida-pseudo exposes them via family.get_recommended_cutoffs)
ecutwfc_rec, ecutrho_rec = pseudo_group.get_recommended_cutoffs(structure=si, unit='Ry')
print(f'PD-A recommended for Si:  ecutwfc={ecutwfc_rec:.1f} Ry  ecutrho={ecutrho_rec:.1f} Ry')

# Build cutoff schedule (8 points, enough to converge for Si)
cutoff_schedule = [
    (i, ecutwfc_rec + i * CUTOFF_STEP_RY)
    for i in range(8)
]
print('\nCutoff schedule (sweep_index, ecutwfc Ry):')
for idx, ecut in cutoff_schedule:
    print(f'  sweep_index={idx}  ecutwfc={ecut:5.1f} Ry  ecutrho={ecut*ECUTRHO_RATIO:5.1f} Ry')


PD-A recommended for Si:  ecutwfc=36.0 Ry  ecutrho=144.0 Ry

Cutoff schedule (sweep_index, ecutwfc Ry):
  sweep_index=0  ecutwfc= 36.0 Ry  ecutrho=144.0 Ry
  sweep_index=1  ecutwfc= 41.0 Ry  ecutrho=164.0 Ry
  sweep_index=2  ecutwfc= 46.0 Ry  ecutrho=184.0 Ry
  sweep_index=3  ecutwfc= 51.0 Ry  ecutrho=204.0 Ry
  sweep_index=4  ecutwfc= 56.0 Ry  ecutrho=224.0 Ry
  sweep_index=5  ecutwfc= 61.0 Ry  ecutrho=244.0 Ry
  sweep_index=6  ecutwfc= 66.0 Ry  ecutrho=264.0 Ry
  sweep_index=7  ecutwfc= 71.0 Ry  ecutrho=284.0 Ry


In [15]:
# Smoke test = the FIRST cutoff sweep point (sweep_index=0).
# This dict prefigures BuilderTags.to_workchain_extras().
# In production, builder.py assembles this from BuilderTags pydantic.

KINDEX_FOR_CUTOFF_SWEEP = 5  # locked moderate kmesh during cutoff sweep (PLAN §5.2)

sweep_idx, ecutwfc = cutoff_schedule[0]
ecutrho = ecutwfc * ECUTRHO_RATIO

# Bands: Si has 8 valence electrons -> n_occ=4. Add >=50% empty bands so HOMO/LUMO is computable.
n_occ = 4
nbnd = n_occ + max(4, (n_occ + 1) // 2)  # = 8 for Si

workchain_extras = {
    # --- sweep state ---
    'round_number': 1,
    'sweep_axis': 'cutoff',
    'sweep_index': sweep_idx,
    # --- pseudo / family ---
    'pseudo_family': pseudo_group.label,
    'pseudo_selection_reason': 'pd_a_default',
    # --- numerics decision cache ---
    'ecutwfc': ecutwfc,
    'ecutrho': ecutrho,
    'cutoff_source': 'family_recommended',
    'kindex': KINDEX_FOR_CUTOFF_SWEEP,  # locked during cutoff sweep
    # --- physics decision cache ---
    'metallicity_guess': 'insulator',  # Si is a known insulator (would be ML-predicted in production)
    'smearing_type': 'cold',
    'degauss': 0.01,
    'nspin': 1,
    'magnetic_state_decision': 'nonmagnetic',
    'soc_enabled': False,
    # --- lifecycle (will be moved to 'submitted' by apply_tags) ---
    'lifecycle_stage': 'created',
    # --- audit ---
    'is_smoke_test': True,
}

group_paths = [
    'sweep/smoke-test.r1',
    'phase/a',
    'calc_type/scf',
    f'pseudo/{pseudo_group.label}',
    f'structure/smoke-test/si-bulk-diamond-a543',
    f'sweep_axis/cutoff/sweep_index/{sweep_idx}',
    'lifecycle/created',
]

print(f'sweep_index={sweep_idx}, ecutwfc={ecutwfc} Ry, kindex(locked)={KINDEX_FOR_CUTOFF_SWEEP}')
print(f'nbnd={nbnd} (n_occ={n_occ}, +50% empty bands)')


sweep_index=0, ecutwfc=36.0 Ry, kindex(locked)=5
nbnd=8 (n_occ=4, +50% empty bands)


In [17]:
# Use ElectronicType.METAL to force occupations='smearing' (since degauss=0.01 is
# universal in Phase 1 — no insulator/metal branch). Then override degauss / smearing
# to match our globally fixed values, and override nbnd so output_band has empty bands.

from aiida import orm as _orm  # alias to avoid shadowing the global `orm`

builder = PwBaseWorkChain.get_builder_from_protocol(
    code=code,
    structure=si,
    protocol='moderate',
    electronic_type=ElectronicType.METAL,
    spin_type=SpinType.NONE,
    overrides={
        'pseudo_family': pseudo_group.label,    # ← top-level key in overrides
        'pw': {
            'parameters': {
                'CONTROL': {
                    'tprnfor': True,
                    'tstress': True,
                },
                'SYSTEM': {
                    'occupations': 'smearing',
                    'smearing': 'cold',
                    'degauss': 0.01,            # PLAN §4: globally fixed
                    'ecutwfc': ecutwfc,         # from cutoff schedule
                    'ecutrho': ecutrho,
                    'nbnd': nbnd,               # PLAN §4: explicit empty bands
                },
                'ELECTRONS': {
                    'mixing_beta': 0.4,
                    'mixing_mode': 'plain',
                    'diagonalization': 'david',
                    'electron_maxstep': 200,
                    'conv_thr': 2e-10,          # Ry/atom, matched to MC3D
                },
            },
        },
    },
)

# Override the protocol's kpoints_distance with a moderate value so we are
# sweeping cutoff at a fixed kmesh density. KINDEX_FOR_CUTOFF_SWEEP=5 maps to
# roughly 0.20 1/A in goldilocks_core.kmesh — for Si this is ~7x7x7 mesh.
builder.kpoints_distance = _orm.Float(0.20)

# Resources: 1 node, 4 MPI; max_wallclock 2h (PLAN §11.3 global cap).
builder.pw.metadata.options.resources = {'num_machines': 1, 'num_mpiprocs_per_machine': 4}
builder.pw.metadata.options.max_wallclock_seconds = 2 * 3600

# Keep workdir for now — we'll explicitly clean after parsing (Cell 9).
builder.clean_workdir = _orm.Bool(False)

print('Builder ready.')
print(f'  ecutwfc={ecutwfc} Ry  ecutrho={ecutrho} Ry  nbnd={nbnd}')
print(f'  kpoints_distance=0.20 1/A  degauss=0.01 Ry  smearing=cold')
print(f'  pseudo_family={pseudo_group.label}')


Builder ready.
  ecutwfc=36.0 Ry  ecutrho=144.0 Ry  nbnd=8
  kpoints_distance=0.20 1/A  degauss=0.01 Ry  smearing=cold
  pseudo_family=PseudoDojo/0.4/PBEsol/SR/standard/upf


In [21]:
# This prefigures aiida_ops.apply_tags() and utils.format_state().
# In production both live in their respective modules.

def format_state(node) -> str:
    """Emoji process state — mirrors aiida-analyser conventions."""
    state = node.process_state.value if node.process_state else 'unknown'
    if state == 'created':   return '🌱 created'
    if state == 'running':   return '⏳ running'
    if state == 'waiting':   return '⏸️  waiting'
    if state == 'killed':    return '💀 killed'
    if state == 'excepted':  return '⚠️  excepted'
    if state == 'finished':
        ec = node.exit_status
        return '✅ finished[0]' if ec == 0 else f'❌ finished[{ec}]'
    return f'❓ {state}'


def apply_tags(node, extras: dict, group_paths: list[str]) -> None:
    """Attach extras + group memberships to a freshly submitted WorkChainNode."""
    node.base.extras.set_many(extras)
    for path in group_paths:
        group, _ = Group.collection.get_or_create(label=path)
        group.add_nodes([node])


def move_lifecycle(node, new_stage: str) -> None:
    """Move node from one lifecycle group to another."""
    old_stage = node.base.extras.get('lifecycle_stage')
    if old_stage == new_stage:
        return
    try:
        Group.collection.get(label=f'lifecycle/{old_stage}').remove_nodes([node])
    except Exception:
        pass
    g, _ = Group.collection.get_or_create(label=f'lifecycle/{new_stage}')
    g.add_nodes([node])
    node.base.extras.set('lifecycle_stage', new_stage)


node = submit(builder)
apply_tags(node, workchain_extras, group_paths)
move_lifecycle(node, 'submitted')

print(f'Submitted PwBaseWorkChain  PK={node.pk}  uuid={node.uuid[:8]}...')
print(f'  state: {format_state(node)}')

# Groups containing this node — anchor on the node first, then join Group via with_node
qb = QueryBuilder()
qb.append(WorkChainNode, filters={'id': node.pk}, tag='n')
qb.append(Group, with_node='n')
print(f'  groups: {[g.label for g in qb.all(flat=True)]}')




Submitted PwBaseWorkChain  PK=34031  uuid=403f4e12...
  state: 🌱 created
  groups: ['sweep/smoke-test.r1', 'phase/a', 'calc_type/scf', 'pseudo/PseudoDojo/0.4/PBEsol/SR/standard/upf', 'structure/smoke-test/si-bulk-diamond-a543', 'sweep_axis/cutoff/sweep_index/0', 'lifecycle/submitted']


In [23]:
# Block until the workchain finishes, then parse the outputs we care about.

import time

# If the SCF already finished from a previous run, this loop exits immediately.
while not node.is_terminated:
    print(f'  [{time.strftime("%H:%M:%S")}] {format_state(node)}')
    time.sleep(30)

print(f'\nFinal state: {format_state(node)}')
move_lifecycle(node, 'finished' if node.is_finished_ok else 'failed')

if node.is_finished_ok:
    p = node.outputs.output_parameters.get_dict()
    band = node.outputs.output_band
    bands = band.get_array('bands')
    n_occ_actual = int(p['number_of_electrons'] // 2)

    # SCF iteration count — key location varies across aiida-qe parser versions
    n_scf = (
        p.get('number_of_iterations')
        or p.get('number_of_scf_iterations')
        or p.get('convergence_info', {}).get('scf_conv', {}).get('n_scf_steps')
        or 'N/A'
    )

    qe_mem = p.get('estimated_ram_per_process', {})
    qe_mem_str = (
        f'{qe_mem.get("value", "N/A")} {qe_mem.get("units", "")}'
        if isinstance(qe_mem, dict) else str(qe_mem)
    )

    print(f'\nKey outputs:')
    print(f'  total_energy             : {p["energy"]:.6f} eV')
    print(f'  fermi_energy             : {p["fermi_energy"]:.4f} eV')
    print(f'  n_scf_iterations         : {n_scf}')
    print(f'  qe_pwscf_wall_seconds    : {p["wall_time_seconds"]:.1f}')
    print(f'  qe_estimated_mem_per_proc: {qe_mem_str}')
    print(f'  bands shape              : {bands.shape}  (k-points x bands)')
    if bands.shape[-1] > n_occ_actual:
        homo = bands[:, n_occ_actual - 1].max()
        lumo = bands[:, n_occ_actual].min()
        print(f'  HOMO / LUMO / gap        : {homo:.3f} / {lumo:.3f} / {lumo - homo:.3f} eV')

    print(f'\nAll output_parameters top-level keys (for schema mapping reference):')
    for k in sorted(p.keys()):
        v = p[k]
        if isinstance(v, (str, int, float, bool)) or v is None:
            print(f'  {k:40s} = {v}')
        elif isinstance(v, dict):
            print(f'  {k:40s} = <dict, keys={list(v.keys())[:5]}{"..." if len(v)>5 else ""}>')
        elif isinstance(v, list):
            print(f'  {k:40s} = <list, len={len(v)}>')
        else:
            print(f'  {k:40s} = <{type(v).__name__}>')

    move_lifecycle(node, 'parsed')



Final state: ✅ finished[0]

Key outputs:
  total_energy             : -230.289252 eV
  fermi_energy             : 6.4836 eV
  n_scf_iterations         : 8
  qe_pwscf_wall_seconds    : 3.3
  qe_estimated_mem_per_proc: 4.32
  bands shape              : (56, 8)  (k-points x bands)
  HOMO / LUMO / gap        : 6.184 / 6.666 / 0.482 eV

All output_parameters top-level keys (for schema mapping reference):
  absolute_magnetization                   = 0.0
  beta_real_space                          = False
  charge_density                           = ./charge-density.dat
  constraint_mag                           = 0
  convergence_info                         = <dict, keys=['scf_conv']>
  creator_name                             = pwscf
  creator_version                          = 7.2
  degauss                                  = 0.136056917253
  dft_exchange_correlation                 = PBESOL
  do_magnetization                         = False
  do_not_use_time_reversal                 = Fals

In [26]:
# Five canonical query patterns. Save as a cookbook for monitor.py / export.py.

# 1) All workchains for a given source structure (uses StructureData fallback)
qb = QueryBuilder()
qb.append(StructureData, filters={'extras.source_id': 'si-bulk-diamond-a543'}, tag='s')
qb.append(WorkChainNode, with_incoming='s', tag='wc')
print(f'(1) workchains using si-bulk-diamond-a543: {qb.count()}')

# 2) All cutoff-sweep workchains in lifecycle=parsed
qb = QueryBuilder()
qb.append(
    WorkChainNode,
    filters={'extras.sweep_axis': 'cutoff', 'extras.lifecycle_stage': 'parsed'},
)
print(f'(2) parsed cutoff-sweep workchains: {qb.count()}')

# 3) All workchains on PD-A with sweep_index=0 (i.e. round 1 of cutoff sweep)
qb = QueryBuilder()
qb.append(
    WorkChainNode,
    filters={
        'extras.pseudo_family': 'PseudoDojo/0.4/PBEsol/SR/standard/upf',
        'extras.sweep_axis': 'cutoff',
        'extras.sweep_index': 0,
    },
    project=['id', 'extras.ecutwfc', 'extras.lifecycle_stage'],
)
print(f'(3) PD-A x cutoff sweep_index=0:')
for pk, ecut, stage in qb.iterall():
    print(f'      PK={pk}  ecutwfc={ecut} Ry  stage={stage}')

# 4) Group-based query: everything in 'lifecycle/finished'
qb = QueryBuilder()
qb.append(Group, filters={'label': 'lifecycle/finished'}, tag='g')
qb.append(WorkChainNode, with_group='g')
print(f'(4) workchains in lifecycle/finished group: {qb.count()}')

# 5) get_source() pattern — fallback from WorkChain to its input StructureData
def get_source(wc_node) -> tuple[str, str]:
    """Return (source_db, source_id), falling back to any input StructureData.

    Robust to WorkChain input namespaces — uses QueryBuilder rather than
    assuming a specific link label like 'structure' or 'pw.structure'.
    Mirrors aiida-analyser's BaseWorkChainAnalyser.get_source().
    """
    wc_extras = wc_node.base.extras
    if all(k in wc_extras for k in ('source_db', 'source_id')):
        return wc_extras.get('source_db'), wc_extras.get('source_id')

    qb = QueryBuilder()
    qb.append(WorkChainNode, filters={'id': wc_node.pk}, tag='wc')
    qb.append(StructureData, with_outgoing='wc')
    structures = qb.all(flat=True)
    if not structures:
        raise ValueError(f'No StructureData input found for node {wc_node.pk}')
    s_extras = structures[0].base.extras
    return s_extras.get('source_db'), s_extras.get('source_id')

src = get_source(node)
print(f'(5) get_source(node) -> {src}')


(1) workchains using si-bulk-diamond-a543: 3
(2) parsed cutoff-sweep workchains: 1
(3) PD-A x cutoff sweep_index=0:
      PK=34017  ecutwfc=36.0 Ry  stage=submitted
      PK=34024  ecutwfc=36.0 Ry  stage=submitted
      PK=34031  ecutwfc=36.0 Ry  stage=parsed
(4) workchains in lifecycle/finished group: 0
(5) get_source(node) -> ('smoke-test', 'si-bulk-diamond-a543')


In [28]:
# Selective HPC cleanup: remove large transient files but keep human-readable
# artifacts and structured outputs on the remote workdir.
#
# DELETE: wfc*.hdf5 (wavefunctions, dominate disk), charge-density.hdf5,
#         spin*.hdf5 (nspin=2), Si.upf etc. inside aiida.save/, pseudo/ dir.
# KEEP:   aiida.in, aiida.out, _aiidasubmit.sh, _scheduler-{stdout,stderr}.txt,
#         out/aiida.save/data-file-schema.xml, out/aiida.xml.
#
# All "KEEP" files are also already in the local AiiDA archive (node.outputs.retrieved),
# so this is "redundancy on HPC" — useful for ad-hoc grep / debug without pulling.

from aiida.orm import CalcJobNode

DELETE_GLOBS = [
    'out/aiida.save/wfc*.hdf5',          # main disk hog
    'out/aiida.save/charge-density.hdf5',
    'out/aiida.save/spin*.hdf5',          # nspin=2 case
    'out/aiida.save/*.upf',               # pseudos copied into save/
    'out/aiida.save/paw.txt',             # PAW dataset (PAW pseudos)
    'pseudo',                              # entire top-level pseudo upload dir
]

calcjobs = [
    d for d in node.called_descendants
    if isinstance(d, CalcJobNode) and 'remote_folder' in d.outputs
]

for cj in calcjobs:
    rf = cj.outputs.remote_folder
    remote_path = rf.get_remote_path()
    print(f'CalcJob PK={cj.pk}  remote={remote_path}')

    with rf.get_authinfo().get_transport() as transport:
        ec, out, err = transport.exec_command_wait(f'du -sh {remote_path}')
        print(f'  size before: {out.strip()}')

        for pat in DELETE_GLOBS:
            cmd = f'cd {remote_path} && rm -rf {pat}'
            ec, _, _ = transport.exec_command_wait(cmd)
            print(f'    rm -rf {pat}  exit={ec}')

        ec, out, _ = transport.exec_command_wait(f'du -sh {remote_path}')
        print(f'  size after:  {out.strip()}')

        ec, out, _ = transport.exec_command_wait(f'ls -la {remote_path}')
        print(f'  remaining contents:\n{out}')

move_lifecycle(node, 'cleaned')
node.base.extras.set('retention_cleaned', True)
print(f'\nFinal lifecycle: {node.base.extras.get("lifecycle_stage")}')


CalcJob PK=34036  remote=/work4/scd/scarf1418/aiida/48/8c/4679-3f17-4cd8-8b1c-946e3ef04e7a
  size before: 
    rm -rf out/aiida.save/wfc*.hdf5  exit=1
    rm -rf out/aiida.save/charge-density.hdf5  exit=1
    rm -rf out/aiida.save/spin*.hdf5  exit=1
    rm -rf out/aiida.save/*.upf  exit=1
    rm -rf out/aiida.save/paw.txt  exit=1
    rm -rf pseudo  exit=1
  size after:  
  remaining contents:


Final lifecycle: cleaned


In [1]:
# Recovery after kernel crash: reload profile + previously submitted node.

from aiida import load_profile
from aiida.orm import (
    Group, QueryBuilder, StructureData, WorkChainNode, load_code, load_group, load_node,
)

load_profile()

NODE_PK = 34031   # the successful smoke test from earlier
node = load_node(NODE_PK)

print(f'Reloaded PwBaseWorkChain PK={node.pk}')
print(f'  state           : {"finished[0]" if node.is_finished_ok else "not finished"}')
print(f'  lifecycle_stage : {node.base.extras.all.get("lifecycle_stage")}')
print(f'  sweep_axis      : {node.base.extras.all.get("sweep_axis")}')
print(f'  sweep_index     : {node.base.extras.all.get("sweep_index")}')
print(f'  ecutwfc         : {node.base.extras.all.get("ecutwfc")} Ry')


Reloaded PwBaseWorkChain PK=34031
  state           : finished[0]
  lifecycle_stage : cleaned
  sweep_axis      : cutoff
  sweep_index     : 0
  ecutwfc         : 36.0 Ry


In [3]:
import time

def t(name):
    class T:
        def __enter__(self): self.s = time.time()
        def __exit__(self, *a): print(f'{name:35s} {time.time() - self.s:6.2f}s')
    return T()

with t('output_parameters.get_dict()'):
    p = node.outputs.output_parameters.get_dict()
with t('output_band.get_array'):
    b = node.outputs.output_band.get_array('bands')
with t('output_trajectory.get_array x2'):
    tr = node.outputs.output_trajectory
    tr.get_array('forces'); tr.get_array('stress')
with t('called_descendants iteration'):
    desc = list(node.called_descendants)
with t('inputs.pw.structure + extras'):
    s = node.inputs.pw.structure
    _ = dict(s.base.extras.all)
with t('inputs.pw.parameters.get_dict()'):
    _ = node.inputs.pw.parameters.get_dict()


output_parameters.get_dict()          0.01s
output_band.get_array                 0.01s
output_trajectory.get_array x2        0.02s
called_descendants iteration          0.01s
inputs.pw.structure + extras          0.01s
inputs.pw.parameters.get_dict()       0.01s


In [4]:
# Preview: turn the smoke test node into a canonical Parquet row.
# This is the prototype of export.py's record-building logic.
# Fields needing data we don't have yet (sacct, convergence across rounds,
# MC3D-specific audit) are set to None to make the schema gaps explicit.

import json
import numpy as np

from ase.data import atomic_numbers


HEAVY_Z = 53  # I and beyond — "heavy" for SOC purposes (PLAN §6.2)
LANTHANIDES = set('La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu'.split())
ACTINIDES   = set('Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr'.split())
PSEUDO_LABEL_FIELDS = ['pseudo_source', 'pseudo_version', 'pseudo_functional',
                      'pseudo_relativistic', 'pseudo_accuracy', 'pseudo_format']


def parse_pseudo_label(label):
    """Parse 'PseudoDojo/0.4/PBEsol/SR/standard/upf' → structured fields."""
    parts = (label or '').split('/')
    if len(parts) == 6:
        return dict(zip(PSEUDO_LABEL_FIELDS, parts))
    return {f: None for f in PSEUDO_LABEL_FIELDS}


def safe(getter, default=None):
    try:
        return getter()
    except Exception:
        return default


def build_record(node) -> dict:
    p = node.outputs.output_parameters.get_dict()
    bands = node.outputs.output_band.get_array('bands')

    pw_inputs = node.inputs.pw
    structure = pw_inputs.structure
    s_extras = dict(structure.base.extras.all)   # ← was: structure.base.extras
    wc_extras = dict(node.base.extras.all)        # ← was: node.base.extras
    pw_params = pw_inputs.parameters.get_dict()

    # Element / structure stats
    elements = sorted(structure.get_symbols_set())
    heavy_elements = sorted([e for e in elements if atomic_numbers.get(e, 0) >= HEAVY_Z])

    # HOMO / LUMO / gap from band eigenvalues
    n_occ = int(p['number_of_electrons'] // 2)
    homo = float(bands[:, n_occ - 1].max()) if bands.shape[-1] >= n_occ else None
    lumo = float(bands[:, n_occ].min()) if bands.shape[-1] > n_occ else None
    gap = (lumo - homo) if (homo is not None and lumo is not None) else None

    # forces / stress from output_trajectory (1 frame for SCF)
    try:
        traj = node.outputs.output_trajectory
        forces_max = float(np.abs(traj.get_array('forces')).max())
        stress_max = float(np.abs(traj.get_array('stress')).max())
    except Exception:
        forces_max = stress_max = None

    # find the underlying PwCalculation
    calc_uuid = next(
        (d.uuid for d in node.called_descendants
         if 'PwCalculation' in str(getattr(d, 'process_type', ''))),
        None,
    )

    # request-side resources (from input metadata.options)
    res = safe(lambda: dict(pw_inputs.metadata.options.resources), {})

    pseudo_parts = parse_pseudo_label(wc_extras.get('pseudo_family'))

    return {
        # === provenance (metadata) ===
        'workchain_uuid':              node.uuid,
        'calculation_uuid':            calc_uuid,
        'aiida_archive_version':       '2.x',
        'goldilocks_data_version':     '0.1.0-smoke',
        'git_sha':                     'smoke',
        'submitted_at':                node.ctime.isoformat(),
        'submitter':                   'yin-junwen',
        'schedule_generator':          'goldilocks_data.cutoff_linear',
        'schedule_generator_version':  '0.1.0',
        'schedule_max_index':          8,

        # === structure features ===
        'source_db':                   s_extras.get('source_db'),
        'source_id':                   s_extras.get('source_id'),
        'formula':                     structure.get_formula(),
        'spacegroup_number':           None,
        'crystal_system':              None,
        'n_atoms':                     len(structure.sites),
        'cell_volume':                 structure.get_cell_volume(),
        'element_set':                 elements,
        'n_electrons_neutral':         p['number_of_electrons'],
        'contains_lanthanide':         bool(set(elements) & LANTHANIDES),
        'contains_actinide':           bool(set(elements) & ACTINIDES),
        'contains_heavy':              bool(heavy_elements),
        'heavy_elements':              heavy_elements,
        'likely_magnetic':             None,
        'magnetic_elements':           [],
        'is_metal_guess':              wc_extras.get('metallicity_guess'),
        'dimensionality_larsen':       None,
        'anisotropy_ratio':            None,

        # === pseudo + numerics ===
        'pseudo_family':               wc_extras.get('pseudo_family'),
        'pseudo_selection_reason':     wc_extras.get('pseudo_selection_reason'),
        **pseudo_parts,
        'ecutwfc':                     wc_extras.get('ecutwfc'),
        'ecutrho':                     wc_extras.get('ecutrho'),
        'k_mesh':                      list(p.get('monkhorst_pack_grid', [])),
        'k_offset':                    list(p.get('monkhorst_pack_offset', [])),
        'k_distance':                  safe(lambda: float(node.inputs.kpoints_distance.value)),
        'k_density_mp':                None,
        'k_linedensity_jarvis':        None,
        'sweep_axis':                  wc_extras.get('sweep_axis'),
        'kindex':                      wc_extras.get('kindex'),
        'k_pra':                       None,
        'n_reduced_kpoints':           p['number_of_k_points'],
        'smearing_type':               wc_extras.get('smearing_type'),
        'degauss':                     wc_extras.get('degauss'),
        'mixing_beta':                 pw_params.get('ELECTRONS', {}).get('mixing_beta'),
        'conv_thr':                    pw_params.get('ELECTRONS', {}).get('conv_thr'),

        # === physics decisions ===
        'nspin':                       wc_extras.get('nspin'),
        'noncolin':                    p.get('non_colinear_calculation', False),
        'lspinorb':                    p.get('spin_orbit_calculation', False),
        'soc_enabled':                 wc_extras.get('soc_enabled'),
        'magnetic_state_decision':     wc_extras.get('magnetic_state_decision'),
        'starting_magnetization_source': None,
        'metallicity_guess':           wc_extras.get('metallicity_guess'),
        'vdw_used':                    False,
        'cutoff_source':               wc_extras.get('cutoff_source'),
        'occupations':                 p.get('occupations'),

        # === physics output (label) ===
        'total_energy':                p['energy'],
        'fermi_energy':                p['fermi_energy'],
        'forces_max':                  forces_max,
        'stress_max':                  stress_max,
        'band_gap_estimate':           gap,
        'total_magnetization':         p.get('total_magnetization'),
        'n_scf_iterations':            p.get('total_number_of_scf_iterations'),
        'final_scf_accuracy':          p.get('energy_accuracy'),
        'exit_status':                 node.exit_status,
        'warnings':                    p.get('warnings', []),

        # === convergence labels (need multiple SCFs; null in smoke test) ===
        'convergence_label_energy':    None,
        'converged_at_index':          None,
        'converged_at_ecutwfc':        None,
        'converged_at_kindex':         None,
        'convergence_status':          None,
        'dE_consecutive_meV_per_atom': None,
        'dF_consecutive_eV_per_A':     None,
        'force_unstable':              None,
        'round_number':                wc_extras.get('round_number'),
        'gamma_pathological':          None,
        'scf_failed':                  not node.is_finished_ok,

        # === resources (request side) ===
        'n_nodes_requested':           res.get('num_machines'),
        'n_mpi_per_node_requested':    res.get('num_mpiprocs_per_machine'),
        'n_omp_threads':               None,
        'walltime_requested_s':        safe(lambda: pw_inputs.metadata.options.max_wallclock_seconds),
        'mem_per_node_requested_mb':   None,

        # === resources (actual side, sacct — n/a in smoke test) ===
        'peak_memory_mb_actual':       None,
        'avg_memory_mb_actual':        None,
        'walltime_actual_s':           None,
        'slurm_exit_code':             None,
        'slurm_state':                 None,
        'qe_estimated_mem_per_proc_mb': p.get('estimated_ram_per_process'),
        'qe_total_ram_summed_mb':      p.get('estimated_ram_total'),
        'qe_pwscf_wall_seconds':       p.get('wall_time_seconds'),
        'memory_efficiency':           None,
        'walltime_efficiency':         None,
        'npool':                       None,
        'nbgrp':                       None,
        'ndiag':                       None,
        'parallelization_strategy':    None,

        # === lifecycle + audit ===
        'lifecycle_stage':             wc_extras.get('lifecycle_stage'),
        'retention_cleaned':           wc_extras.get('retention_cleaned', False),
        'is_smoke_test':               wc_extras.get('is_smoke_test', False),
        'phase':                       'a',  # derived from group membership in production
        'mc3d_pseudo_flagged_suboptimal': s_extras.get('mc3d_pseudo_flagged_suboptimal'),
        'mc3d_afm_likely':             s_extras.get('mc3d_afm_likely'),
        'mc3d_high_pressure':          s_extras.get('mc3d_high_pressure'),
        'mc3d_theoretical_only':       s_extras.get('mc3d_theoretical_only'),
        'mc3d_total_magnetization':    s_extras.get('mc3d_total_magnetization'),
        'mc3d_absolute_magnetization': s_extras.get('mc3d_absolute_magnetization'),
    }


record = build_record(node)
print(f'Built record with {len(record)} fields\n')

# Display grouped by PLAN §6 sub-section for readability
sections = {
    '§6.1 PROVENANCE':        ['workchain_uuid', 'calculation_uuid', 'goldilocks_data_version',
                                'git_sha', 'submitted_at', 'submitter',
                                'schedule_generator', 'schedule_max_index'],
    '§6.2 STRUCTURE':         ['source_db', 'source_id', 'formula', 'n_atoms', 'cell_volume',
                                'element_set', 'n_electrons_neutral', 'heavy_elements',
                                'contains_lanthanide', 'contains_actinide', 'contains_heavy',
                                'is_metal_guess'],
    '§6.3 PSEUDO + NUMERICS': ['pseudo_family', 'pseudo_selection_reason',
                                'pseudo_source', 'pseudo_functional', 'pseudo_relativistic',
                                'ecutwfc', 'ecutrho', 'k_mesh', 'k_offset', 'k_distance',
                                'sweep_axis', 'kindex', 'n_reduced_kpoints',
                                'smearing_type', 'degauss', 'mixing_beta', 'conv_thr'],
    '§6.4 PHYSICS DECISIONS': ['nspin', 'noncolin', 'lspinorb', 'soc_enabled',
                                'magnetic_state_decision', 'metallicity_guess',
                                'cutoff_source', 'occupations'],
    '§6.5 PHYSICS OUTPUT':    ['total_energy', 'fermi_energy', 'band_gap_estimate',
                                'forces_max', 'stress_max', 'total_magnetization',
                                'n_scf_iterations', 'final_scf_accuracy', 'exit_status'],
    '§6.6 CONVERGENCE':       ['convergence_label_energy', 'converged_at_index',
                                'converged_at_ecutwfc', 'converged_at_kindex',
                                'round_number', 'scf_failed'],
    '§6.7 RESOURCES':         ['n_nodes_requested', 'n_mpi_per_node_requested',
                                'walltime_requested_s', 'qe_estimated_mem_per_proc_mb',
                                'qe_total_ram_summed_mb', 'qe_pwscf_wall_seconds',
                                'peak_memory_mb_actual', 'walltime_actual_s'],
    '§6.8 LIFECYCLE + AUDIT': ['lifecycle_stage', 'retention_cleaned', 'is_smoke_test',
                                'phase', 'mc3d_afm_likely'],
}

for section, fields in sections.items():
    print(f'━━━ {section} ━━━')
    for f in fields:
        v = record.get(f, '<MISSING>')
        if isinstance(v, float):
            v = f'{v:.6g}'
        elif isinstance(v, list) and len(str(v)) > 60:
            v = f'<list len={len(v)}>'
        elif v is None:
            v = '∅'
        print(f'  {f:35s} = {v}')
    print()

# Replace the polars block at the end of Cell 10 with this:

import pandas as pd
import os

# Coerce list/dict columns to JSON strings for cross-engine compatibility
record_for_pd = {
    k: json.dumps(v) if isinstance(v, (list, dict)) else v
    for k, v in record.items()
}
df = pd.DataFrame([record_for_pd])
print(f'\npandas DataFrame: {len(df)} row × {df.shape[1]} columns')

# Write a smoke partition
out_dir = (
    'data/processed/v0.1.0-smoke/'
    f'structure_id={record["source_db"]}-{record["source_id"]}/'
)
os.makedirs(out_dir, exist_ok=True)
out_path = out_dir + 'part-0.parquet'
df.to_parquet(out_path, engine='pyarrow', index=False)

print(f'Wrote: {out_path}')

# Round-trip read to verify
df_read = pd.read_parquet(out_path)
print(f'Read back: {len(df_read)} row × {df_read.shape[1]} columns ✓')


Built record with 111 fields

━━━ §6.1 PROVENANCE ━━━
  workchain_uuid                      = 403f4e12-b274-49e2-8b94-c8084f9e480c
  calculation_uuid                    = ∅
  goldilocks_data_version             = 0.1.0-smoke
  git_sha                             = smoke
  submitted_at                        = 2026-05-08T14:39:03.339566+01:00
  submitter                           = yin-junwen
  schedule_generator                  = goldilocks_data.cutoff_linear
  schedule_max_index                  = 8

━━━ §6.2 STRUCTURE ━━━
  source_db                           = smoke-test
  source_id                           = si-bulk-diamond-a543
  formula                             = Si2
  n_atoms                             = 2
  cell_volume                         = 40.0258
  element_set                         = ['Si']
  n_electrons_neutral                 = 8
  heavy_elements                      = []
  contains_lanthanide                 = False
  contains_actinide                   = False

In [7]:
# Smoke-test cleanup: delete WorkChainNodes + their input StructureData + the groups.
# delete_nodes cascades to all linked children (CalcJobs / outputs / retrieved FolderData).

from aiida.orm import Group, QueryBuilder, StructureData, WorkChainNode
from aiida.tools import delete_nodes

# --- Step 1: collect what to delete ---
# Smoke-test workchains (have extras.is_smoke_test=True)
qb = QueryBuilder()
qb.append(WorkChainNode, filters={'extras.is_smoke_test': True}, project='id')
wc_pks = [r[0] for r in qb.all()]

# Smoke-test structures (extras.source_db='smoke-test')
qb = QueryBuilder()
qb.append(StructureData, filters={'extras.source_db': 'smoke-test'}, project='id')
struct_pks = [r[0] for r in qb.all()]

top_pks = wc_pks + struct_pks
print(f'Top-level to delete: {len(wc_pks)} workchains + {len(struct_pks)} structures = {len(top_pks)}')

# Step 2: dry-run
to_delete, _ = delete_nodes(top_pks, dry_run=True)
print(f'(dry-run) cascaded deletion would touch {len(to_delete)} nodes total')
print(f'PKs preview: {sorted(to_delete)[:20]}{"..." if len(to_delete) > 20 else ""}')



05/08/2026 03:45:48 PM <64195> aiida.delete: [REPORT] 28 Node(s) marked for deletion
05/08/2026 03:45:48 PM <64195> aiida.delete: [REPORT] This was a dry run, exiting without deleting anything


Top-level to delete: 3 workchains + 1 structures = 4
(dry-run) cascaded deletion would touch 28 nodes total
PKs preview: [34011, 34017, 34018, 34019, 34022, 34023, 34024, 34025, 34026, 34029, 34030, 34031, 34032, 34033, 34036, 34037, 34038, 34039, 34040, 34041]...


In [8]:
# Step 3: real delete (cascades to all 28 nodes)
to_delete, was_deleted = delete_nodes(top_pks, dry_run=False)
print(f'Deletion executed={was_deleted}, removed {len(to_delete)} nodes')

# Step 4: drop now-empty groups
GROUPS_TO_DELETE = [
    'sweep/smoke-test.r1',
    'phase/a',
    'calc_type/scf',
    'pseudo/PseudoDojo/0.4/PBEsol/SR/standard/upf',
    'structure/smoke-test/si-bulk-diamond-a543',
    'sweep_axis/cutoff/sweep_index/0',
    'lifecycle/created',
    'lifecycle/submitted',
    'lifecycle/finished',
    'lifecycle/parsed',
    'lifecycle/cleaned',
]

for label in GROUPS_TO_DELETE:
    try:
        g = Group.collection.get(label=label)
        n = g.count()
        if n > 0:
            print(f'  skip {label}: still has {n} nodes')
            continue
        Group.collection.delete(g.pk)
        print(f'  deleted group {label}')
    except Exception as e:
        print(f'  skip {label}: {e}')


05/08/2026 03:46:32 PM <64195> aiida.delete: [REPORT] 28 Node(s) marked for deletion
05/08/2026 03:46:32 PM <64195> aiida.delete: [REPORT] Starting node deletion...
05/08/2026 03:46:32 PM <64195> aiida.delete: [REPORT] Deletion of nodes completed.


Deletion executed=True, removed 28 nodes
  deleted group sweep/smoke-test.r1
  deleted group phase/a
  deleted group calc_type/scf
  deleted group pseudo/PseudoDojo/0.4/PBEsol/SR/standard/upf
  deleted group structure/smoke-test/si-bulk-diamond-a543
  deleted group sweep_axis/cutoff/sweep_index/0
  deleted group lifecycle/created
  deleted group lifecycle/submitted
  deleted group lifecycle/finished
  deleted group lifecycle/parsed
  deleted group lifecycle/cleaned
